# AgencyOS — Phase 3 GPU preflight

This notebook only checks the runtime. It does **not** download a model, load a dataset, call an AI API, or start training.

## Before running
1. Keep this notebook **private**.
2. In notebook settings, select **GPU** if one is available.
3. Run the single code cell below.
4. A passing result is required before Phase 4. Stop the session after the check.

Do not attach client data, production exports, credentials, or confidential project files.

In [ ]:
import platform, sys
print('AgencyOS Phase 3 — GPU preflight')
print('Python:', platform.python_version())
if sys.version_info < (3, 10) or sys.version_info >= (3, 12):
    raise RuntimeError('Use Python 3.10 or 3.11 (3.11 preferred).')

try:
    import torch
except Exception as exc:
    raise RuntimeError(f'PyTorch import failed: {exc}') from exc

print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda or 'CPU-only')
if not torch.cuda.is_available():
    raise RuntimeError('BLOCKED: no CUDA GPU is available. Enable GPU in notebook settings or retry later.')

print('Visible CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gib = p.total_memory / (1024 ** 3)
    print(f'GPU {i}: {p.name} | VRAM {gib:.2f} GiB | compute capability {p.major}.{p.minor}')
    if gib < 12:
        raise RuntimeError(f'BLOCKED: GPU {i} has less than 12 GiB VRAM.')

probe = torch.ones((1024, 1024), dtype=torch.float16, device='cuda:0')
print('CUDA allocation test: PASS | checksum:', float(probe.sum().item()))
del probe
torch.cuda.synchronize()
torch.cuda.empty_cache()
print('\nRESULT: PASS — runtime meets the Phase 3 minimum.')
print('This does not prove a particular model fits; model-specific sizing is still required.')